# ConFit Fine-Tuning — Notebook 4: `all-mpnet-base-v2` on Kaggle (P100)

Fine-tunes `sentence-transformers/all-mpnet-base-v2` on the synthetic triplet dataset generated in NB3.

| Setting | Value |
|---|---|
| Accelerator | **P100 (16 GB VRAM)** |
| Loss | `TripletLoss` (cosine, margin=0.5) |
| Extra negatives | Cross-skill negatives (30%) |
| Split | 90 / 10 stratified by level |
| Precision | FP16 |

## Before Running
1. Go to **Add Data** → **Your Datasets** → add `confit-synthetic-pairs` (the dataset you uploaded).
2. Confirm the path in the **Config** cell matches your dataset slug.
3. Set **Accelerator → GPU P100** in Notebook Settings.


In [1]:
# ── Environment — MUST run before any torch import ────────────────────────────
# sentence-transformers TripletLoss is not compatible with nn.DataParallel
# (the sentence_features iterable is consumed by the first replica, leaving
#  the second replica with StopIteration). Force single-GPU to avoid this.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only GPU 0, even on 2×T4 nodes
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # synchronous errors — better tracebacks

# ── Install Dependencies ───────────────────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", *args], check=True)

pip("install", "-q",
    "sentence-transformers>=3.0.0",
    "datasets",
    "accelerate",
    "scikit-learn")

print("Dependencies ready.")


Dependencies ready.


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import TripletLoss
from sentence_transformers.evaluation import TripletEvaluator

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Verify single-GPU setup
print(f"PyTorch   : {torch.__version__}")
print(f"Visible GPUs (after CUDA_VISIBLE_DEVICES=0): {torch.cuda.device_count()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device    : {props.name}")
    print(f"VRAM      : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute   : sm_{props.major}{props.minor}")
    assert torch.cuda.device_count() == 1, \
        "CUDA_VISIBLE_DEVICES='0' should restrict to 1 GPU. Restart kernel and re-run cell 1 first."
else:
    print("WARNING: No GPU visible — ensure accelerator is enabled in Kaggle settings.")


KeyboardInterrupt: 

In [ ]:
# ── Config — edit DATASET_SLUG if your Kaggle dataset name differs ─────────────
DATASET_SLUG = "confit-synthetic-pairs"       # ← your Kaggle dataset slug
PAIRS_FILE   = f"/kaggle/input/{DATASET_SLUG}/synthetic_pairs.jsonl"
OUTPUT_DIR   = "/kaggle/working/confit-mpnet-v1"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Model
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

# Hyper-parameters (tuned for P100 16 GB)
BATCH_SIZE   = 64      # safe for all-mpnet on P100 with FP16
EPOCHS       = 4
LR           = 2e-5
WARMUP_RATIO = 0.10    # 10 % of total steps
TRIPLET_MARGIN = 0.5   # cosine distance margin
CROSS_RATIO    = 0.30  # fraction of extra cross-skill negatives
VAL_SPLIT      = 0.10  # 90/10 stratified split
EVAL_STEPS     = 200   # evaluate & checkpoint every N steps
LOG_STEPS      = 50

print("Config loaded.")
print(f"  Pairs file : {PAIRS_FILE}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Batch size : {BATCH_SIZE}  |  Epochs: {EPOCHS}  |  LR: {LR}")


## Step 1 — Load & Inspect Data


In [ ]:
# ── Load synthetic pairs ──────────────────────────────────────────────────────
records = []
with open(PAIRS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            # Keep only valid, complete rows
            if (rec.get("status") == "ok"
                    and rec.get("anchor")
                    and rec.get("positive")
                    and rec.get("hard_negative")):
                records.append(rec)
        except json.JSONDecodeError:
            pass

df = pd.DataFrame(records)
print(f"Valid pairs loaded : {len(df):,}")
print(f"Unique anchors     : {df['anchor'].nunique():,}")
print(f"Unique skills      : {df['skill'].nunique():,}")
print(f"\nLevel distribution:")
print(df["level"].value_counts().to_string())

# Sanity-check a sample
print("\n─── Sample Record ───────────────────────────────────────────────────────")
s = df.sample(1).iloc[0]
print(f"  Skill  : {s['skill']}  [{s['level']}]")
print(f"  Anchor : {s['anchor'][:100]}...")
print(f"  Pos    : {s['positive'][:100]}...")
print(f"  Neg    : {s['hard_negative'][:100]}...")


## Step 2 — Build Triplets (LLM negatives + Cross-Skill negatives)


In [ ]:
# ── Build (anchor, positive, negative) triplets ───────────────────────────────
#
# Two sources of negatives per anchor:
#   1. LLM hard negative  — same skill, wrong depth/style   (high signal)
#   2. Cross-skill negative — different skill's positive     (easy discrimination)
#
# Having both types prevents the model from solely learning keyword patterns and
# forces it to understand depth/specificity differences.

all_positives = df["positive"].tolist()

triplets = []

# 1 — Direct LLM triplets (one per row)
for _, row in df.iterrows():
    triplets.append({
        "anchor":   row["anchor"],
        "positive": row["positive"],
        "negative": row["hard_negative"],
        "level":    row["level"],   # kept for stratified split, dropped later
    })

# 2 — Cross-skill triplets (CROSS_RATIO fraction of total)
n_cross = int(len(df) * CROSS_RATIO)
for _ in range(n_cross):
    row = df.sample(1).iloc[0]
    # Sample a positive from a DIFFERENT skill
    candidate = random.choice(all_positives)
    while candidate == row["positive"]:
        candidate = random.choice(all_positives)
    triplets.append({
        "anchor":   row["anchor"],
        "positive": row["positive"],
        "negative": candidate,
        "level":    row["level"],
    })

random.shuffle(triplets)
triplet_df = pd.DataFrame(triplets)

print(f"Total triplets         : {len(triplet_df):,}")
print(f"  LLM hard negatives   : {len(df):,}")
print(f"  Cross-skill negatives: {n_cross:,}")
print(f"\nLevel distribution in triplet pool:")
print(triplet_df["level"].value_counts().to_string())


## Step 3 — Stratified Train / Val Split (by level)


In [ ]:
# ── Stratified split — each seniority level represented in both splits ────────
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=SEED)
train_idx, val_idx = next(sss.split(triplet_df, triplet_df["level"]))

train_df = triplet_df.iloc[train_idx].copy()
val_df   = triplet_df.iloc[val_idx].copy()

# Drop the helper 'level' column — trainer only needs anchor/positive/negative
MODEL_COLS = ["anchor", "positive", "negative"]

train_dataset = Dataset.from_pandas(train_df[MODEL_COLS].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[MODEL_COLS].reset_index(drop=True))

print(f"Train triplets : {len(train_dataset):,}")
print(f"Val triplets   : {len(val_dataset):,}")
print(f"\nTrain level distribution:")
print(train_df["level"].value_counts().to_string())
print(f"\nVal level distribution:")
print(val_df["level"].value_counts().to_string())


## Step 4 — Model, Loss & Evaluator


In [ ]:
# ── Load base model ───────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME)
print(f"Base model loaded  : {MODEL_NAME}")
print(f"Embedding dim      : {model.get_sentence_embedding_dimension()}")
print(f"Max seq length     : {model.max_seq_length}")
print(f"Trainable params   : {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f} M")

# ── Loss ──────────────────────────────────────────────────────────────────────
# TripletLoss(cosine) maximises sim(a,p) and minimises sim(a,n).
# Triplet margin: loss = max(0, sim(a,n) - sim(a,p) + margin)
loss = TripletLoss(
    model=model,
    triplet_margin=TRIPLET_MARGIN,
)
print(f"\nLoss: TripletLoss (cosine, margin={TRIPLET_MARGIN})")

# ── Evaluator — measures % of val triplets ranked correctly ──────────────────
evaluator = TripletEvaluator(
    anchors=val_df["anchor"].tolist(),
    positives=val_df["positive"].tolist(),
    negatives=val_df["negative"].tolist(),
    name="val",
    show_progress_bar=True,
)
print("Evaluator: TripletEvaluator (cosine accuracy on val set)")


## Step 5 — Training Arguments & Trainer


In [ ]:
# ── Training Arguments ────────────────────────────────────────────────────────
steps_per_epoch = len(train_dataset) // BATCH_SIZE
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print(f"Steps/epoch     : {steps_per_epoch}")
print(f"Total steps     : {total_steps}")
print(f"Warmup steps    : {warmup_steps}")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",

    # Precision
    fp16=True,

    # Eval & checkpointing
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=3,            # keep only the 3 best checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="val_cosine_accuracy",  # TripletEvaluator metric
    greater_is_better=True,

    # Logging
    logging_steps=LOG_STEPS,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",              # set to "tensorboard" if you want TensorBoard

    # Determinism
    seed=SEED,
    data_seed=SEED,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
    evaluator=evaluator,
)

print("\nTrainer configured. Ready to train.")


## Step 6 — Train 🚀

Expected runtime on **P100**: ~35–50 min for 4 epochs on ~10k triplets.


In [ ]:
# ── Run Training ──────────────────────────────────────────────────────────────
train_result = trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Total steps        : {train_result.global_step}")
print(f"  Training loss      : {train_result.training_loss:.4f}")
print(f"  Runtime            : {train_result.metrics.get('train_runtime', 0) / 60:.1f} min")
print(f"  Samples/sec        : {train_result.metrics.get('train_samples_per_second', 0):.1f}")


## Step 7 — Post-Training Evaluation


In [ ]:
# ── Final triplet accuracy on val set ────────────────────────────────────────
eval_results = evaluator(model)
print("Final Val Triplet Evaluation")
print("=" * 40)
for k, v in eval_results.items():
    print(f"  {k:<35}: {v:.4f}")


In [ ]:
# ── Cosine Similarity Distribution Plot ──────────────────────────────────────
# Visualise positive vs. negative similarity gap on a 200-sample probe.
probe = val_df.sample(min(200, len(val_df)), random_state=SEED)

anchors_enc   = model.encode(probe["anchor"].tolist(),   convert_to_tensor=True, show_progress_bar=False)
positives_enc = model.encode(probe["positive"].tolist(), convert_to_tensor=True, show_progress_bar=False)
negatives_enc = model.encode(probe["negative"].tolist(), convert_to_tensor=True, show_progress_bar=False)

from torch.nn.functional import cosine_similarity
pos_sims = cosine_similarity(anchors_enc, positives_enc).cpu().numpy()
neg_sims = cosine_similarity(anchors_enc, negatives_enc).cpu().numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pos_sims, bins=30, alpha=0.65, label="Anchor ↔ Positive", color="#2ecc71")
ax.hist(neg_sims, bins=30, alpha=0.65, label="Anchor ↔ Negative", color="#e74c3c")
ax.axvline(pos_sims.mean(), color="#27ae60", linestyle="--", linewidth=1.5, label=f"Pos mean={pos_sims.mean():.3f}")
ax.axvline(neg_sims.mean(), color="#c0392b", linestyle="--", linewidth=1.5, label=f"Neg mean={neg_sims.mean():.3f}")
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Count")
ax.set_title("Cosine Similarity Distribution — Validation Probe (n=200)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/similarity_distribution.png", dpi=150)
plt.show()
print(f"Mean gap (pos - neg): {pos_sims.mean() - neg_sims.mean():.4f}  (higher = better separation)")


## Step 8 — Save Model & Export


In [ ]:
# ── Save best model (load_best_model_at_end=True already applied) ─────────────
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final"
model.save(FINAL_MODEL_DIR)
print(f"Model saved to : {FINAL_MODEL_DIR}")

# ── List output files ─────────────────────────────────────────────────────────
import os
print("\nOutput directory contents:")
for root, dirs, files in os.walk(FINAL_MODEL_DIR):
    for fn in files:
        fp = os.path.join(root, fn)
        size_kb = os.path.getsize(fp) / 1024
        print(f"  {fp.replace(FINAL_MODEL_DIR, '.'):<50}  {size_kb:>8.1f} KB")


In [ ]:
# ── Quick Smoke Test — encode 3 sentences and check similarity ────────────────
test_anchor   = "Develop and maintain production REST APIs using Python and FastAPI."
test_positive = "Built and deployed 4 FastAPI microservices handling 50k requests/day with 99.9% uptime."
test_negative = "Helped the team with some Python coding tasks and reviewed FastAPI documentation."

vecs = model.encode([test_anchor, test_positive, test_negative])
from sklearn.metrics.pairwise import cosine_similarity as sk_cos
sims = sk_cos(vecs[:1], vecs[1:])[0]

print("Smoke Test — Cosine Similarities")
print(f"  Anchor ↔ Positive (should be HIGH): {sims[0]:.4f}")
print(f"  Anchor ↔ Negative (should be LOW) : {sims[1]:.4f}")
print(f"  Gap (pos - neg)                   : {sims[0] - sims[1]:.4f}")
assert sims[0] > sims[1], "WARNING: Model ranking is inverted — check training!"
print("\nSmoke test PASSED.")


In [ ]:
# ── Package model as zip for easy download ────────────────────────────────────
import shutil
zip_path = shutil.make_archive(
    base_name=f"{OUTPUT_DIR}/confit-mpnet-v1",
    format="zip",
    root_dir=FINAL_MODEL_DIR,
)
zip_size_mb = os.path.getsize(zip_path) / 1e6
print(f"Model zip created: {zip_path}  ({zip_size_mb:.1f} MB)")
print("\nDownload via Kaggle → Output → confit-mpnet-v1/confit-mpnet-v1.zip")
print("\n" + "=" * 60)
print("NB4 COMPLETE")
print("=" * 60)
print("Next step: download the zip, extract to app/models/confit-mpnet-v1/")
print("Then update vector_client.py to point to the new model path.")
